## Config

In [1]:
# ============================================================
# CONFIG  — edit only this cell
# ============================================================

PRETRAINED_ADAPTER_PATH = "/kaggle/input/training-with-unsloth-no-lm-head-v2/sft_adapter"  # Unsloth-trained LoRA
DATASET_PATH            = "/kaggle/input/datasets/rohit4118/nemotron/crystal_math_rewritten.csv"
BASE_MODEL_NAME         = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"   # HF slug → written into adapter_config
BASE_MODEL_PATH         = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OFFLINE_PKGS            = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"

# ── Output paths (all derived here — never redefined later) ───────────────────
OUTPUT_DIR             = "/kaggle/working"
GRPO_RUN_DIR           = f"{OUTPUT_DIR}/grpo_output"
ADAPTER_DIR            = f"{OUTPUT_DIR}/grpo_adapter"          # FIX(doc): was inside training cell
SUBMISSION_ADAPTER_DIR = f"{OUTPUT_DIR}/submission_adapter"
SUBMISSION_ZIP         = f"{OUTPUT_DIR}/submission.zip"

# ── Data ──────────────────────────────────────────────────────────────────────
SEED            = 42
GRPO_SAMPLES    = 450
PROMPT_SUFFIX   = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

print({
    "PRETRAINED_ADAPTER_PATH": PRETRAINED_ADAPTER_PATH,
    "DATASET_PATH":            DATASET_PATH,
    "BASE_MODEL_NAME":         BASE_MODEL_NAME,
    "GRPO_SAMPLES":            GRPO_SAMPLES,
    "ADAPTER_DIR":             ADAPTER_DIR,
})

{'PRETRAINED_ADAPTER_PATH': '/kaggle/input/training-with-unsloth-no-lm-head-v2/sft_adapter', 'DATASET_PATH': '/kaggle/input/datasets/rohit4118/nemotron/crystal_math_rewritten.csv', 'BASE_MODEL_NAME': 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16', 'GRPO_SAMPLES': 450, 'ADAPTER_DIR': '/kaggle/working/grpo_adapter'}


In [2]:
# ============================================================
# EARLY PACKAGE INSTALL — must run BEFORE any imports
# Installs transformers 5.5.4 from hastws/nemotron-tf55
# Keep Docker defaults for huggingface_hub + peft (no path validation issues)
# ============================================================
import subprocess, sys, glob, os

def _install_wheel(pattern, label):
    wheels = sorted(glob.glob(pattern, recursive=True))
    if not wheels:
        raise RuntimeError(f"Wheel not found: {label}. Add dataset hastws/nemotron-tf55 via Kaggle UI.")
    w = wheels[-1]
    print(f"  {label}: installing {os.path.basename(w)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", w], check=True)

print("Installing transformers 5.5.4 (before any imports)...")
_install_wheel("/kaggle/input/**/transformers-5.5*.whl", "transformers")
print("Done — transformers 5.5.4 installed (hub + peft unchanged).")

Installing transformers 5.5.4 (before any imports)...
  transformers: installing transformers-5.5.4-py3-none-any.whl
Processing /kaggle/input/nemotron-tf55/transformers-5.5.4-py3-none-any.whl
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Done — transformers 5.5.4 installed (hub + peft unchanged).


## Triton Installation

In [3]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
print("Custom target added:", target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))


Found Triton wheels: ['/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7b9ed0ec44d0>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


## ptxas-Blackwell Patch

In [4]:
import sys, os, shutil, stat

sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")

ptxas_src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"
ptxas_dst = "/tmp/ptxas-blackwell"

if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    src_bin = os.path.dirname(ptxas_src)
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_dst
    os.environ["TRITON_PTXAS_PATH"]           = ptxas_dst

    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: "12.0"

print("Training environment fixes applied.")


Training environment fixes applied.


## TRL & Mamba Dependencies

In [5]:
import glob, importlib.util, os, subprocess, sys, types

def sh(cmd: str, check: bool = True):
    print("+", cmd)
    return subprocess.run(cmd, shell=True, check=check)

def find_spec(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def recursive_wheels(pattern: str):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

import torch
py_tag   = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
abi_tag  = "cxx11abiTRUE" if torch.compiled_with_cxx11_abi() else "cxx11abiFALSE"

print(f"Python {sys.version} | torch {torch.__version__} | CUDA {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required.")

def pick_best(wheels):
    exact   = [w for w in wheels if py_tag in w and f"torch{torch_mm}" in w and abi_tag in w]
    if exact:   return exact[-1]
    py_only = [w for w in wheels if py_tag in w]
    return py_only[-1] if py_only else None

if not find_spec("datasets"):
    w = pick_best(recursive_wheels("datasets-*.whl"))
    if w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
if not find_spec("trl"):
    w = pick_best(recursive_wheels("trl-*.whl"))
    if w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
for pkg, pat in [("multiprocess", "multiprocess-*.whl"),
                 ("dill",         "dill-*.whl"),
                 ("xxhash",       "xxhash-*.whl")]:
    if not find_spec(pkg):
        w = pick_best(recursive_wheels(pat))
        if w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"', check=False)

# FIX(fork): direct assignment (not setdefault) — overwrites any stale partial
# submodule from a prior cell run in the same kernel session.
# Must be done BEFORE installing/importing mamba_ssm.
for _mod in ["mamba_ssm.modules.mamba3",
             "mamba_ssm.ops.cute",
             "mamba_ssm.ops.cute.mamba3",
             "mamba_ssm.ops.cute.mamba3.mamba3_step_fn"]:
    sys.modules[_mod] = types.ModuleType(_mod)
sys.modules["mamba_ssm.modules.mamba3"].Mamba3 = None

if not find_spec("mamba_ssm"):
    causal_w = pick_best(recursive_wheels("causal*conv1d*.whl"))
    mamba_w  = pick_best(recursive_wheels("mamba_ssm-*.whl"))
    if causal_w: sh(f'{sys.executable} -m pip install --no-index --no-deps "{causal_w}"')
    if mamba_w:  sh(f'{sys.executable} -m pip install --no-index --no-deps "{mamba_w}"')

# Install trl if needed (for training mode)
if True:
    try:
        import trl
        print(f"trl already installed: {trl.__version__}")
    except ImportError:
        # Try offline install first, then online
        offline_path = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"
        if os.path.exists(offline_path):
            subprocess.run(f"pip install --no-index --find-links={offline_path} trl", shell=True)
        else:
            subprocess.run("pip install trl", shell=True)
        import trl
        print(f"trl installed: {trl.__version__}")


import datasets, trl, mamba_ssm
print(f"datasets: {datasets.__version__}  trl: {trl.__version__}  mamba_ssm: {mamba_ssm.__version__}")
print("✓ Dependencies ready")


Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0] | torch 2.10.0+cu128 | CUDA True
+ /usr/bin/python3 -m pip install --no-index --no-deps "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
+ /usr/bin/python3 -m pip install --no-index --no-deps "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Looking in links: /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/trl-0.29.1-py3-none-any.whl
Processing /kaggle/input/datasets/dennisfong/nvidia-nem

## Transformers 5.5.4 (GRPO KV-cache Fix)

Install transformers >= 5.3 to fix the Nemotron GRPO KV-cache bug (~38 tok/s vs ~2 tok/s).
All wheels come from `hastws/nemotron-tf55` — make sure this dataset is added in Kaggle UI.

In [6]:
# Packages were already installed in the early-install cell above.
# Just verify versions and set USE_TRUST_REMOTE flag.
import transformers, huggingface_hub
print(f"transformers: {transformers.__version__}  huggingface_hub: {huggingface_hub.__version__}")

# trust_remote_code=True — vLLM-compatible layer names, patched in model cell.
USE_TRUST_REMOTE = True
print(f"trust_remote_code={USE_TRUST_REMOTE}")

transformers: 5.5.4  huggingface_hub: 1.8.0
trust_remote_code=True


## Load Base Model + Pre-trained LoRA

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

import os as _os
import huggingface_hub.utils._validators as _hf_validators
_orig_val = _hf_validators.validate_repo_id
_hf_validators.validate_repo_id = lambda rid: None if (isinstance(rid, str) and rid.startswith('/') and _os.path.exists(rid)) else _orig_val(rid)

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH, device_map="cuda:0",
    trust_remote_code=True, torch_dtype=torch.bfloat16)
print("Base model loaded.")

# ---- Patch custom code for transformers 5.5+ ----
_orig_prep = base_model.prepare_inputs_for_generation
def _patch_prep(*args, **kwargs):
    cp = kwargs.get('cache_position', None)
    if cp is None:
        input_ids = args[0] if args else kwargs.get('input_ids')
        dev = input_ids.device
        pkv = kwargs.get('past_key_values', None)
        plen = pkv.get_seq_length() if (pkv and hasattr(pkv, 'get_seq_length')) else (pkv[0][0].shape[2] if pkv else 0)
        kwargs = dict(kwargs)
        kwargs['cache_position'] = torch.arange(plen, plen + input_ids.shape[1], device=dev)
    return _orig_prep(*args, **kwargs)
base_model.prepare_inputs_for_generation = _patch_prep
if hasattr(base_model, '_validate_model_kwargs'):
    base_model._validate_model_kwargs = lambda self, *a, **kw: None
print("Patched prepare_inputs_for_generation + validate_model_kwargs.")

# ---- Load pre-trained Unsloth LoRA via PeftModel.from_pretrained ----
# This lets PEFT handle the key mapping from Unsloth's saved state dict
# (base_model.model.backbone.*) to the HuggingFace model structure.
model = PeftModel.from_pretrained(
    base_model, PRETRAINED_ADAPTER_PATH,
    is_trainable=True,
)
model = model.to("cuda")
assert next(model.parameters()).is_cuda, "Model is still on CPU!"
model.print_trainable_parameters()
print("Pre-trained LoRA loaded and ready for GRPO.")

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Base model loaded.
Patched prepare_inputs_for_generation + validate_model_kwargs.


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:912: RuntimeWarning: target_parameters=['mlp.experts.gate_up_proj', 'mlp.experts.down_proj'] were set but no parameter was matched.
  warnings.warn(


trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228
Pre-trained LoRA loaded and ready for GRPO.


## Data Preparation (100 Rows)

In [8]:
import pandas as pd
from datasets import Dataset as HFDataset

# Load data
df = pd.read_csv(DATASET_PATH)

# The code previously executed successfully with these column names, so we keep them for reading.
# Note: If your CSV actually uses 'prompt' instead of 'problem', change row["problem"] to row["prompt"].
df = df.dropna(subset=["solution", "problem", "answer"])
df = df.sample(n=min(GRPO_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print(f"Loaded {len(df)} rows.")

# FIX: TRL GRPO requires the prompt column to be named "prompt", not "problem".
records = []
for _, row in df.iterrows():
    records.append({
        "prompt": str(row["problem"]) + PROMPT_SUFFIX,  # Changed key from "problem" to "prompt"
        "answer": str(row["answer"]),
    })

dataset = HFDataset.from_list(records)
print(f"Dataset ready — {len(dataset)} records")
print(f"  Sample prompt [:100]: {records[0]['prompt'][:100]}")
print(f"  Sample answer       : {records[0]['answer']}")

Loaded 450 rows.
Dataset ready — 450 records
  Sample prompt [:100]: Find the smallest positive real number \(c\) such that there exists a 2024-element subset \(A\) of \
  Sample answer       : 7


## Reward Functions

In [9]:
import re

# ── extract_response (from document) ──────────────────────────────────────────
# Defensive helper: TRL normally passes plain strings but some versions pass
# list-of-dicts [{role, content}]. This handles both safely.
def extract_response(comp) -> str:
    if isinstance(comp, list):
        return comp[-1]["content"] if comp else ""
    return str(comp)

# ── correctness_reward_func ───────────────────────────────────────────────────
# FIX(doc):  removed `prompts` as first positional arg — TRL calls
#            reward_func(completions=..., **dataset_cols) without passing
#            `prompts` positionally → TypeError: missing required argument.
# FIX(fork): `answer` is a LIST (TRL broadcasts it once per completion).
#            str(["42","42"]) = "['42','42']" → never matches → reward = 0.
#            Fixed by iterating zip(completions, answers).
def correctness_reward_func(completions, answer, **kwargs):
    """
    +1.0 if the last \boxed{} in the completion matches ground truth, else 0.0.
    `answer` is a list — one ground-truth string per completion.
    """
    rewards  = []
    # Guard: accept both list (TRL ≥ 0.12) and scalar (older TRL)
    answers  = answer if isinstance(answer, (list, tuple)) else [answer] * len(completions)
    for comp, gt in zip(completions, answers):
        text    = extract_response(comp)
        matches = re.findall(r"\\boxed\{([^}]*)\}", text)
        pred    = matches[-1].strip() if matches else ""
        rewards.append(1.0 if pred == str(gt).strip() else 0.0)
    return rewards

# ── format_reward_func ────────────────────────────────────────────────────────
# FIX(doc): removed `prompts` positional arg (same reason as above)
def format_reward_func(completions, **kwargs):
    """
    +0.2 if <think>…</think> present (chain-of-thought structure)
    +0.1 if \boxed{} present
    """
    rewards = []
    for comp in completions:
        text = extract_response(comp)
        r    = 0.0
        if "<think>" in text and "</think>" in text:
            r += 0.2
        if re.search(r"\\boxed\{", text):
            r += 0.1
        rewards.append(r)
    return rewards

print("Reward functions ready.")
print("  correctness_reward_func : +1.0 exact boxed match, 0.0 otherwise")
print("  format_reward_func      : +0.2 think tags, +0.1 boxed present")


Reward functions ready.
  correctness_reward_func : +1.0 exact boxed match, 0.0 otherwise
  format_reward_func      : +0.2 think tags, +0.1 boxed present


## GRPO Training

In [10]:
import time, inspect
from trl import GRPOConfig, GRPOTrainer

# ── Detect valid GRPOConfig param names at runtime ───────────────────────────
# The generation-length arg was renamed across TRL versions:
#   TRL <= 0.11 : max_completion_length
#   TRL 0.12-0.14: max_new_tokens
#   TRL >= 0.15 : max_completion_length  (reverted)
# temperature / use_vllm also moved in/out of GRPOConfig across versions.
# Hard-coding either name breaks on a different TRL version — inspect instead.
_valid = set(inspect.signature(GRPOConfig.__init__).parameters)
print(f"TRL version : {__import__('trl').__version__}")
print(f"GRPOConfig generation-related params: "
      f"{sorted(p for p in _valid if any(k in p for k in ('max','temp','gen','token','complet','vllm')))}")

def _pick(candidates, value):
    """Return {first_matching_param_name: value}, or {} if none found."""
    for name in candidates:
        if name in _valid:
            return {name: value}
    return {}

_gen_len = _pick(["max_completion_length", "max_new_tokens"], 7680)
_prompt_len = _pick(["max_prompt_length"],512)
_temp    = _pick(["temperature"], 0.9)
_vllm    = _pick(["use_vllm"], False)
print(f"  gen-length kwarg : {_gen_len}")
print(f"  temperature kwarg: {_temp}")
print(f"  use_vllm kwarg   : {_vllm}")

# ---- DEBUG: manual generation test BEFORE training ----
print("\n=== DEBUG: Manual generation test ===")
test_prompt = dataset[0]["prompt"]
test_answer = dataset[0]["answer"]
print(f"Prompt[:200]: {test_prompt[:200]}")
print(f"Ground truth: {test_answer}")

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=7680,
        do_sample=True,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated = full_text[len(test_prompt):]  # just the new part
print(f"\n--- Generated (last 500 chars) ---")
print(generated[-500:])
print(f"\n--- Reward check ---")
print(f"  format_reward: {format_reward_func([generated])}")
print(f"  correctness:   {correctness_reward_func([generated], answer=[test_answer])}")
print(f"=== DEBUG END ===\n")
# ---- DEBUG END ----

# ── GRPOConfig ────────────────────────────────────────────────────────────────
# NemotronHForCausalLM does NOT support gradient checkpointing — must disable.
# TRL GRPO is generation-step based, not epoch-based — use max_steps + steps_per_generation.
training_args = GRPOConfig(
    output_dir                    = GRPO_RUN_DIR,
    max_steps                     = 28,   # 28 generation steps ≈ 450 samples / (4×2×2) effective batch
    num_train_epochs              = 1,
    per_device_train_batch_size   = 2,
    gradient_accumulation_steps   = 4,
    learning_rate                 = 5e-6,
    lr_scheduler_type             = "cosine",
    warmup_ratio                  = 0.1,
    #max_completion_length         = 512,
    num_generations               = 2,    # 2 completions per prompt for advantage estimation
    generation_batch_size         = 2,    # must be divisible by num_generations
    logging_steps                 = 1,    # log every step to debug reward/loss
    log_completions               = True,  # print sample completions to debug
    save_strategy                 = "no",
    bf16                          = True,
    gradient_checkpointing        = False,  # Nemotron + PEFT does NOT support gradient checkpointing
    dataloader_num_workers        = 4,
    remove_unused_columns         = False,
    seed                          = SEED,
    report_to                     = "none",
    generation_kwargs             = {"use_cache": True, "do_sample": True},
    **_gen_len,   # max_completion_length OR max_new_tokens — whichever this TRL accepts
    **_temp,      # temperature — present in some TRL versions only
    **_vllm,      # use_vllm   — present in some TRL versions only
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [correctness_reward_func, format_reward_func],
    args             = training_args,
    train_dataset    = dataset,
)

print(f"  Dataset size    : {len(dataset)}")
print(f"  Effective batch : {training_args.per_device_train_batch_size} × {training_args.gradient_accumulation_steps} × {training_args.num_generations} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * training_args.num_generations}")

print("\nStarting GRPO training…")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"\nGRPO done in {elapsed/60:.1f} min")

TRL version : 0.29.1
GRPOConfig generation-related params: ['average_tokens_across_devices', 'chat_template_kwargs', 'ds3_gather_for_generation', 'generation_batch_size', 'generation_kwargs', 'hub_token', 'include_num_input_tokens_seen', 'log_completions', 'log_completions_hub_repo', 'mask_truncated_completions', 'max_completion_length', 'max_grad_norm', 'max_steps', 'max_tool_calling_iterations', 'num_completions_to_print', 'num_generations', 'num_generations_eval', 'sapo_temperature_neg', 'sapo_temperature_pos', 'steps_per_generation', 'temperature', 'use_vllm', 'vllm_enable_sleep_mode', 'vllm_gpu_memory_utilization', 'vllm_group_port', 'vllm_importance_sampling_cap', 'vllm_importance_sampling_correction', 'vllm_importance_sampling_mode', 'vllm_max_model_length', 'vllm_mode', 'vllm_model_impl', 'vllm_server_base_url', 'vllm_server_host', 'vllm_server_port', 'vllm_server_timeout', 'vllm_structured_outputs_regex', 'vllm_tensor_parallel_size']
  gen-length kwarg : {'max_completion_lengt

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



--- Generated (last 500 chars) ---


--- Reward check ---
  format_reward: [0.0]
  correctness:   [0.0]
=== DEBUG END ===



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


  Dataset size    : 450
  Effective batch : 2 × 4 × 2 = 16

Starting GRPO training…


...

╭──────────────────────────────────────────────────── Step 26 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                              ┃ Completion ┃ correctness_reward_func ┃ format_reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Let $P$-$ABCD$ and $Q$-$ABCD$ be    │            │                    0.00 │               0.00 │      0.00 │ │
│ │ two regular square pyramids, and    │            │                         │                    │           │ │
│ │ $\angle P A Q=90^{\circ}$. Point    │            │                         │                    │           │ │
│ │ $M$ lies on segment $A C$ such that │            │                         │                    │           │ │
│ │ $C M=3 A M$. Denote the angle       │            │                         │                    │           │ │
│ │ between the skew lines $P M$ and $Q │            │                         │                    │           │ │
│ │ B$ by $\theta$. Then the maximum    │            │                         │                    │           │ │
│ │ possible value of $\cos \theta$ is  │            │                         │                    │           │ │
│ │ $\qquad$. The answer should be      │            │                         │                    │           │ │
│ │ given in the form $\frac{m}{n}$,    │            │                         │                    │           │ │
│ │ where $m$ and $n$ are coprime. Find │            │                         │                    │           │ │
│ │ the final value of $m + n$.         │            │                         │                    │           │ │
│ │ Please put your final answer inside │            │                         │                    │           │ │
│ │ `\boxed{}`. For example:            │            │                         │                    │           │ │
│ │ `\boxed{your answer}`               │            │                         │                    │           │ │
│ ├─────────────────────────────────────┼────────────┼─────────────────────────┼────────────────────┼───────────┤ │
│ │ Let $P$-$ABCD$ and $Q$-$ABCD$ be    │            │                    0.00 │               0.00 │      0.00 │ │
│ │ two regular square pyramids, and    │            │                         │                    │           │ │
│ │ $\angle P A Q=90^{\circ}$. Point    │            │                         │                    │           │ │
│ │ $M$ lies on segment $A C$ such that │            │                         │                    │           │ │
│ │ $C M=3 A M$. Denote the angle       │            │                         │                    │           │ │
│ │ between the skew lines $P M$ and $Q │            │                         │                    │           │ │
│ │ B$ by $\theta$. Then the maximum    │            │                         │                    │           │ │
│ │ possible value of $\cos \theta$ is  │            │                         │                    │           │ │
│ │ $\qquad$. The answer should be      │            │                         │                    │           │ │
│ │ given in the form $\frac{m}{n}$,    │            │                         │                    │           │ │
│ │ where $m$ and $n$ are coprime. Find │            │                         │                    │           │ │
│ │ the final value of $m + n$.         │            │                         │                    │           │ │
│ │ Please put your final answer inside │            │                         │                    │           │ │
│ │ `\boxed{}`. For example:            │            │                         │                    │           │ │
│ │ `\boxed{your answer}`               │            │  

╭──────────────────────────────────────────────────── Step 27 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                              ┃ Completion ┃ correctness_reward_func ┃ format_reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ It is known that one plane divides  │            │                    0.00 │               0.00 │      0.00 │ │
│ │ the space (three-dimensional) into  │            │                         │                    │           │ │
│ │ two parts, two parallel planes      │            │                         │                    │           │ │
│ │ divide the space into three parts,  │            │                         │                    │           │ │
│ │ and two intersecting planes divide  │            │                         │                    │           │ │
│ │ the space into four parts; Consider │            │                         │                    │           │ │
│ │ the planes where the six faces of   │            │                         │                    │           │ │
│ │ the cube \(ABCD - A_1B_1C_1D_1\)    │            │                         │                    │           │ │
│ │ are located, and the planes where   │            │                         │                    │           │ │
│ │ the four faces of the tetrahedron   │            │                         │                    │           │ │
│ │ \(BA_1C_1D_1\) are located. How     │            │                         │                    │           │ │
│ │ many parts does these ten planes    │            │                         │                    │           │ │
│ │ divide the space into?              │            │                         │                    │           │ │
│ │ Please put your final answer inside │            │                         │                    │           │ │
│ │ `\boxed{}`. For example:            │            │                         │                    │           │ │
│ │ `\boxed{your answer}`               │            │                         │                    │           │ │
│ ├─────────────────────────────────────┼────────────┼─────────────────────────┼────────────────────┼───────────┤ │
│ │ It is known that one plane divides  │            │                    0.00 │               0.00 │      0.00 │ │
│ │ the space (three-dimensional) into  │            │                         │                    │           │ │
│ │ two parts, two parallel planes      │            │                         │                    │           │ │
│ │ divide the space into three parts,  │            │                         │                    │           │ │
│ │ and two intersecting planes divide  │            │                         │                    │           │ │
│ │ the space into four parts; Consider │            │                         │                    │           │ │
│ │ the planes where the six faces of   │            │                         │                    │           │ │
│ │ the cube \(ABCD - A_1B_1C_1D_1\)    │            │                         │                    │           │ │
│ │ are located, and the planes where   │            │                         │                    │           │ │
│ │ the four faces of the tetrahedron   │            │                         │                    │           │ │
│ │ \(BA_1C_1D_1\) are located. How     │            │                         │                    │           │ │
│ │ many parts does these ten planes    │            │                         │                    │           │ │
│ │ divide the space into?              │            │                         │                    │           │ │
│ │ Please put your final answer inside │            │  

╭──────────────────────────────────────────────────── Step 28 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                              ┃ Completion ┃ correctness_reward_func ┃ format_reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Two persons $A$ and $B$ play a game │            │                    0.00 │               0.00 │      0.00 │ │
│ │ with two decks of 26 cards each,    │            │                         │                    │           │ │
│ │ numbered from 1 to 26. The cards    │            │                         │                    │           │ │
│ │ are distributed randomly between    │            │                         │                    │           │ │
│ │ the two players, and each player    │            │                         │                    │           │ │
│ │ arranges their 26 cards in a deck.  │            │                         │                    │           │ │
│ │ At each turn, they reveal the top   │            │                         │                    │           │ │
│ │ card of their deck, and the player  │            │                         │                    │           │ │
│ │ with the higher number gets a       │            │                         │                    │           │ │
│ │ point. If player $A$ wins the game  │            │                         │                    │           │ │
│ │ but the sum of the numbers on $A$'s │            │                         │                    │           │ │
│ │ cards ($S_a$) is less than the sum  │            │                         │                    │           │ │
│ │ of the numbers on $B$'s cards       │            │                         │                    │           │ │
│ │ ($S_b$), find the maximum value of  │            │                         │                    │           │ │
│ │ $S_b - S_a$.                        │            │                         │                    │           │ │
│ │ Please put your final answer inside │            │                         │                    │           │ │
│ │ `\boxed{}`. For example:            │            │                         │                    │           │ │
│ │ `\boxed{your answer}`               │            │                         │                    │           │ │
│ ├─────────────────────────────────────┼────────────┼─────────────────────────┼────────────────────┼───────────┤ │
│ │ Two persons $A$ and $B$ play a game │            │                    0.00 │               0.00 │      0.00 │ │
│ │ with two decks of 26 cards each,    │            │                         │                    │           │ │
│ │ numbered from 1 to 26. The cards    │            │                         │                    │           │ │
│ │ are distributed randomly between    │            │                         │                    │           │ │
│ │ the two players, and each player    │            │                         │                    │           │ │
│ │ arranges their 26 cards in a deck.  │            │                         │                    │           │ │
│ │ At each turn, they reveal the top   │            │                         │                    │           │ │
│ │ card of their deck, and the player  │            │                         │                    │           │ │
│ │ with the higher number gets a       │            │                         │                    │           │ │
│ │ point. If player $A$ wins the game  │            │                         │                    │           │ │
│ │ but the sum of the numbers on $A$'s │            │                         │                    │           │ │
│ │ cards ($S_a$) is less than the sum  │            │  

╭──────────────────────────────────────────────────── Step 28 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                              ┃ Completion ┃ correctness_reward_func ┃ format_reward_func ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ Two persons $A$ and $B$ play a game │            │                    0.00 │               0.00 │      0.00 │ │
│ │ with two decks of 26 cards each,    │            │                         │                    │           │ │
│ │ numbered from 1 to 26. The cards    │            │                         │                    │           │ │
│ │ are distributed randomly between    │            │                         │                    │           │ │
│ │ the two players, and each player    │            │                         │                    │           │ │
│ │ arranges their 26 cards in a deck.  │            │                         │                    │           │ │
│ │ At each turn, they reveal the top   │            │                         │                    │           │ │
│ │ card of their deck, and the player  │            │                         │                    │           │ │
│ │ with the higher number gets a       │            │                         │                    │           │ │
│ │ point. If player $A$ wins the game  │            │                         │                    │           │ │
│ │ but the sum of the numbers on $A$'s │            │                         │                    │           │ │
│ │ cards ($S_a$) is less than the sum  │            │                         │                    │           │ │
│ │ of the numbers on $B$'s cards       │            │                         │                    │           │ │
│ │ ($S_b$), find the maximum value of  │            │                         │                    │           │ │
│ │ $S_b - S_a$.                        │            │                         │                    │           │ │
│ │ Please put your final answer inside │            │                         │                    │           │ │
│ │ `\boxed{}`. For example:            │            │                         │                    │           │ │
│ │ `\boxed{your answer}`               │            │                         │                    │           │ │
│ ├─────────────────────────────────────┼────────────┼─────────────────────────┼────────────────────┼───────────┤ │
│ │ Two persons $A$ and $B$ play a game │            │                    0.00 │               0.00 │      0.00 │ │
│ │ with two decks of 26 cards each,    │            │                         │                    │           │ │
│ │ numbered from 1 to 26. The cards    │            │                         │                    │           │ │
│ │ are distributed randomly between    │            │                         │                    │           │ │
│ │ the two players, and each player    │            │                         │                    │           │ │
│ │ arranges their 26 cards in a deck.  │            │                         │                    │           │ │
│ │ At each turn, they reveal the top   │            │                         │                    │           │ │
│ │ card of their deck, and the player  │            │                         │                    │           │ │
│ │ with the higher number gets a       │            │                         │                    │           │ │
│ │ point. If player $A$ wins the game  │            │                         │                    │           │ │
│ │ but the sum of the numbers on $A$'s │            │                         │                    │           │ │
│ │ cards ($S_a$) is less than the sum  │            │  


GRPO done in 5.9 min


## Save Adapter

In [11]:
import os

os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

for fname in ["adapter_config.json", "adapter_model.safetensors"]:
    fpath = os.path.join(ADAPTER_DIR, fname)
    if os.path.exists(fpath):
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
    else:
        print(f"  WARNING: {fname} not found!")


Adapter saved to /kaggle/working/grpo_adapter
  adapter_config.json: 0.0 MB
  adapter_model.safetensors: 3373.4 MB


## Package submission.zip

In [12]:
import json, os, shutil, zipfile

os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]
print(f"Packaging adapter from: {ADAPTER_DIR}")

for fname in required_files:
    src = os.path.join(ADAPTER_DIR, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"  Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME   # HF slug
cfg["inference_mode"]           = True
cfg["lora_dropout"]             = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        zf.write(os.path.join(SUBMISSION_ADAPTER_DIR, fname), fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB  →  {SUBMISSION_ZIP}")
print("Done! Ready to submit.")


Packaging adapter from: /kaggle/working/grpo_adapter
  Copied adapter_config.json (0.0 MB)
  Copied adapter_model.safetensors (3373.4 MB)
  Added adapter_config.json
  Added adapter_model.safetensors

submission.zip: 3094.6 MB  →  /kaggle/working/submission.zip
Done! Ready to submit.


## Verification

In [13]:
import os

checks = {
    "Adapter input path exists"       : os.path.exists(PRETRAINED_ADAPTER_PATH),
    "GRPO samples loaded"             : len(dataset) == GRPO_SAMPLES,
    "adapter_config.json saved"       : os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")),
    "adapter_model.safetensors saved" : os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")),
    "submission.zip created"          : os.path.exists(SUBMISSION_ZIP),
    "inference_mode = True"           : cfg.get("inference_mode") == True,
    "lora_dropout = 0.0"              : cfg.get("lora_dropout") == 0.0,
    "base_model is HF slug"           : cfg.get("base_model_name_or_path") == BASE_MODEL_NAME,
}

print("\n" + "="*55)
print("VERIFICATION SUMMARY")
print("="*55)
all_pass = True
for check, passed in checks.items():
    if not passed: all_pass = False
    print(f"  {'✓ PASS' if passed else '✗ FAIL'}: {check}")
print("="*55)
print("All checks PASSED ✓" if all_pass else "Some checks FAILED — see above ✗")



VERIFICATION SUMMARY
  ✓ PASS: Adapter input path exists
  ✓ PASS: GRPO samples loaded
  ✓ PASS: adapter_config.json saved
  ✓ PASS: adapter_model.safetensors saved
  ✓ PASS: submission.zip created
  ✓ PASS: inference_mode = True
  ✓ PASS: lora_dropout = 0.0
  ✓ PASS: base_model is HF slug
All checks PASSED ✓
